In [1]:
import json

candidates = []

with open("F:\\autorecruit-\\data_forensic _files\\candidates.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        candidates.append(json.loads(line))

In [2]:
candidate_texts = []
candidate_ids = []

for c in candidates:

    skills = " ".join(
        [
            s["name"]
            for s in c["skills"]
        ]
    )

    career = " ".join(
        [
            job["description"]
            for job in c["career_history"]
        ]
    )

    summary = c["profile"]["summary"]

    title = c["profile"]["current_title"]

    text = f"""
    {title}

    {summary}

    {skills}

    {career}
    """

    candidate_texts.append(text)

    candidate_ids.append(
        c["candidate_id"]
    )

In [3]:
candidate_texts

["\n    Backend Engineer\n\n    Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.\n\n    Tailwind NLP Image Classification Fine-tuning LLMs Weights & Biases Speech Recognition Photoshop TTS LoRA Apache Beam AWS Flask BentoML Milvus GANs Statistical Modeling GCP\n\n    Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry 

In [4]:
candidate_ids

['CAND_0000001',
 'CAND_0000002',
 'CAND_0000003',
 'CAND_0000004',
 'CAND_0000005',
 'CAND_0000006',
 'CAND_0000007',
 'CAND_0000008',
 'CAND_0000009',
 'CAND_0000010',
 'CAND_0000011',
 'CAND_0000012',
 'CAND_0000013',
 'CAND_0000014',
 'CAND_0000015',
 'CAND_0000016',
 'CAND_0000017',
 'CAND_0000018',
 'CAND_0000019',
 'CAND_0000020',
 'CAND_0000021',
 'CAND_0000022',
 'CAND_0000023',
 'CAND_0000024',
 'CAND_0000025',
 'CAND_0000026',
 'CAND_0000027',
 'CAND_0000028',
 'CAND_0000029',
 'CAND_0000030',
 'CAND_0000031',
 'CAND_0000032',
 'CAND_0000033',
 'CAND_0000034',
 'CAND_0000035',
 'CAND_0000036',
 'CAND_0000037',
 'CAND_0000038',
 'CAND_0000039',
 'CAND_0000040',
 'CAND_0000041',
 'CAND_0000042',
 'CAND_0000043',
 'CAND_0000044',
 'CAND_0000045',
 'CAND_0000046',
 'CAND_0000047',
 'CAND_0000048',
 'CAND_0000049',
 'CAND_0000050',
 'CAND_0000051',
 'CAND_0000052',
 'CAND_0000053',
 'CAND_0000054',
 'CAND_0000055',
 'CAND_0000056',
 'CAND_0000057',
 'CAND_0000058',
 'CAND_0000059

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = model.encode(
    candidate_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

c:\Users\kambl\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

In [6]:
embeddings.size

38400000

In [8]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=500,
    min_samples=20
)

labels = clusterer.fit_predict(
    embeddings
)

In [9]:
import pandas as pd

cluster_df = pd.DataFrame({
    "candidate_id": candidate_ids,
    "cluster": labels
})

In [10]:
cluster_df["cluster"].value_counts()

cluster
 0    68821
 1    25000
 2     5000
 3      998
-1      181
Name: count, dtype: int64

In [11]:
import pandas as pd

quality_rows = []

for c in candidates:

    s = c["redrob_signals"]

    quality_rows.append({

        "candidate_id":
        c["candidate_id"],

        "saved":
        s["saved_by_recruiters_30d"],

        "views":
        s["profile_views_received_30d"],

        "appearances":
        s["search_appearance_30d"],

        "connections":
        s["connection_count"],

        "endorsements":
        s["endorsements_received"],

        "completion":
        s["profile_completeness_score"],

        "interview":
        s["interview_completion_rate"]

    })

quality_df = pd.DataFrame(
    quality_rows
)

In [12]:
quality_df

,candidate_id,saved,views,appearances,connections,endorsements,completion,interview
0,CAND_0000001,4,23,249,356,35,86.9,0.71
1,CAND_0000002,10,7,107,179,3,78.7,0.62
2,CAND_0000003,4,1,28,19,46,31.9,0.86
3,CAND_0000004,8,3,5,485,22,28.5,0.35
4,CAND_0000005,1,12,67,300,14,84.6,0.74
...,...,...,...,...,...,...,...,...
99995,CAND_0099996,7,52,98,325,12,41.4,0.87
99996,CAND_0099997,0,42,49,292,10,42.7,0.38
99997,CAND_0099998,26,84,187,442,66,75.9,0.79
99998,CAND_0099999,3,81,102,345,10,49.1,0.49


In [13]:
from sklearn.preprocessing import StandardScaler

features = [

    "saved",
    "views",
    "appearances",
    "connections",
    "endorsements",
    "completion",
    "interview"
]

X = StandardScaler().fit_transform(
    quality_df[features]
)

In [14]:
from sklearn.decomposition import PCA

pca = PCA(
    n_components=1
)

quality_df[
    "quality_score"
] = pca.fit_transform(X)

In [15]:
quality_df[
    "quality_score"
].describe()

count    1.000000e+05
mean    -1.250555e-17
std      1.409218e+00
min     -3.145835e+00
25%     -8.920829e-01
50%     -2.753696e-01
75%      5.528335e-01
max      1.771064e+01
Name: quality_score, dtype: float64

In [16]:
quality_df.sort_values(
    "quality_score",
    ascending=False
).head(50)

,candidate_id,saved,views,appearances,connections,endorsements,completion,interview,quality_score
93546,CAND_0093547,57,168,1340,989,242,62.6,0.97,17.710640
68350,CAND_0068351,66,179,1347,1255,100,94.5,0.95,16.470047
92277,CAND_0092278,1,327,1417,1307,223,65.4,0.78,15.639309
6566,CAND_0006567,76,167,1049,262,194,58.6,0.93,15.368776
80765,CAND_0080766,27,374,1105,767,182,92.0,0.88,15.336506
8424,CAND_0008425,55,260,1136,1421,97,57.2,0.77,15.046598
84818,CAND_0084819,50,99,1056,1497,174,95.2,0.82,14.412755
37979,CAND_0037980,65,249,748,1659,87,69.3,0.89,14.154163
83878,CAND_0083879,64,274,574,1486,140,59.6,0.83,14.068289
18498,CAND_0018499,16,75,1291,1839,189,98.0,0.80,13.572639


In [64]:
honeypot_scores = []
data = pd.read_csv("candidate_behavior_scores_full.csv")

for idx, c in enumerate(candidates):
    score = 0
    
    # Get the corresponding row from data
    candidate_id = c["candidate_id"]
    candidate_data = data[data["candidate_id"] == candidate_id]
    
    if candidate_data.empty:
        honeypot_scores.append(0)
        continue
    
    row = candidate_data.iloc[0]
    
    if row["date_anomaly"]:
        score += 3
    
    if row["education_score"] < 1:
        score += 2
    
    if pd.notna(row.get("skill_anomaly")) and row["skill_anomaly"]:
        score += 2
    
    if pd.notna(row.get("skill_desc_similarity")):
        if row["skill_desc_similarity"] < 0.05:
            score += 3
        elif row["skill_desc_similarity"] < 0.10:
            score += 2
        elif row["skill_desc_similarity"] < 0.20:
            score += 1
    
    honeypot_scores.append({
        "candidate_id" : c["candidate_id"],
        "honeypot_score" : score})

C:\Users\kambl\AppData\Local\Temp\ipykernel_8360\3851820416.py:2: DtypeWarning: Columns (0: skill_anomaly_x, 1: skill_anomaly_y) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("candidate_behavior_scores_full.csv")


In [65]:
honeypot_scores_df = pd.DataFrame(honeypot_scores)

honeypot_scores_df.describe()

,honeypot_score
count,100000.000000
mean,0.725930
std,1.176227
min,0.000000
25%,0.000000
50%,0.000000
75%,2.000000
max,8.000000


In [66]:
honeypot_scores_df

,candidate_id,honeypot_score
0,CAND_0000001,1
1,CAND_0000002,0
2,CAND_0000003,0
3,CAND_0000004,3
4,CAND_0000005,2
...,...,...
99995,CAND_0099996,0
99996,CAND_0099997,0
99997,CAND_0099998,2
99998,CAND_0099999,0


In [56]:
honeypot_labels = []
for s in honeypot_scores:

    if s["score"] == 0:
        honeypot_labels.append({
            "candidate_id": s["candidate_id"],
            "honey_pot_labels": "trusted"
        })

    elif s["score"] <= 2:
        honeypot_labels.append({
            "candidate_id": s["candidate_id"],
            "honey_pot_labels": "minor_issue"
        })

    elif s["score"] <= 5:
        honeypot_labels.append({
            "candidate_id": s["candidate_id"],
            "honey_pot_labels": "suspicious"
        })
    else:
        honeypot_labels.append({
            "candidate_id": s["candidate_id"],
            "honey_pot_labels": "high_risk"
        })
    
    

In [67]:
honeypot_labels_df = pd.DataFrame(honeypot_labels)
honeypot_labels_df

,candidate_id,honey_pot_labels
0,CAND_0000001,minor_issue
1,CAND_0000002,trusted
2,CAND_0000003,trusted
3,CAND_0000004,suspicious
4,CAND_0000005,minor_issue
...,...,...
99995,CAND_0099996,trusted
99996,CAND_0099997,trusted
99997,CAND_0099998,minor_issue
99998,CAND_0099999,trusted


In [68]:
honeypot_scores_df = honeypot_scores_df.merge(
    honeypot_labels_df,
    on="candidate_id",
    how="left"
)

In [69]:
honeypot_scores_df

,candidate_id,honeypot_score,honey_pot_labels
0,CAND_0000001,1,minor_issue
1,CAND_0000002,0,trusted
2,CAND_0000003,0,trusted
3,CAND_0000004,3,suspicious
4,CAND_0000005,2,minor_issue
...,...,...,...
99995,CAND_0099996,0,trusted
99996,CAND_0099997,0,trusted
99997,CAND_0099998,2,minor_issue
99998,CAND_0099999,0,trusted


In [70]:
honeypot_scores_df.columns.tolist()

['candidate_id', 'honeypot_score', 'honey_pot_labels']

In [71]:
honeypot_scores_df.head(20)

,candidate_id,honeypot_score,honey_pot_labels
0,CAND_0000001,1,minor_issue
1,CAND_0000002,0,trusted
2,CAND_0000003,0,trusted
3,CAND_0000004,3,suspicious
4,CAND_0000005,2,minor_issue
5,CAND_0000006,6,high_risk
6,CAND_0000007,1,minor_issue
7,CAND_0000008,1,minor_issue
8,CAND_0000009,3,suspicious
9,CAND_0000010,0,trusted


In [75]:
behavior_df = pd.read_csv("candidate_behavior_scores_full.csv")

behavior_df = behavior_df.merge(
    honeypot_scores_df[["candidate_id", "honeypot_score"]],
    on="candidate_id",
    how="left"
)
behavior_df = behavior_df.merge(
    honeypot_scores_df[["candidate_id", "honey_pot_labels"]],
    on="candidate_id",
    how="left"
)

behavior_df.to_csv("candidate_behavior_scores_full.csv", index=False)

print(behavior_df.head())
print(behavior_df.shape)

C:\Users\kambl\AppData\Local\Temp\ipykernel_8360\1590361298.py:1: DtypeWarning: Columns (0: skill_anomaly_x, 1: skill_anomaly_y) have mixed types. Specify dtype option on import or set low_memory=False.
  behavior_df = pd.read_csv("candidate_behavior_scores_full.csv")


   candidate_id  profile_completeness  open_to_work  response_rate  \
0  CAND_0000001                  86.9          True           0.34   
1  CAND_0000002                  78.7          True           0.29   
2  CAND_0000003                  31.9         False           0.46   
3  CAND_0000004                  28.5         False           0.26   
4  CAND_0000005                  84.6          True           0.37   

   response_time  views  applications  connections  search_appearance  saved  \
0          177.8     23             2          356                249      4   
1          171.6      7             1          179                107     10   
2          119.4      1             9           19                 28      4   
3          104.1      3             9          485                  5      8   
4          116.7     12             2          300                 67      1   

   ...  skill_desc_similarity_x  date_anomaly  education_score_y  \
0  ...                 0.15674

In [76]:
df = pd.read_csv("candidate_behavior_scores_full.csv")

C:\Users\kambl\AppData\Local\Temp\ipykernel_8360\3533008024.py:1: DtypeWarning: Columns (0: skill_anomaly_x, 1: skill_anomaly_y) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("candidate_behavior_scores_full.csv")


In [78]:
cols_to_keep = [

    "candidate_id",

    # behavior
    "behavior_score",
    "availability_score",
    "responsiveness_score",
    "demand_score",
    "reliability_score",

    # trust
    "date_anomaly",
    "education_score",
    "skill_desc_similarity",
    "skill_anomaly_y",

    # profile
    "profile_completeness",

    # honeypot
    "honeypot_score",
    "honey_pot_labels"
]

master_df = df[cols_to_keep].copy()

In [79]:
master_df

,candidate_id,behavior_score,availability_score,responsiveness_score,demand_score,reliability_score,date_anomaly,education_score,skill_desc_similarity,skill_anomaly_y,profile_completeness,honeypot_score,honey_pot_labels
0,CAND_0000001,0.489693,0.84,0.347500,0.076872,0.658,False,1.0,0.156749,NaN,86.9,1,minor_issue
1,CAND_0000002,NaN,0.84,0.319143,0.082477,NaN,False,1.0,0.252431,NaN,78.7,0,trusted
2,CAND_0000003,NaN,0.00,0.494071,0.029561,NaN,False,1.0,0.223676,NaN,31.9,0,trusted
3,CAND_0000004,NaN,0.08,0.370464,0.053078,NaN,False,0.5,0.137554,NaN,28.5,3,suspicious
4,CAND_0000005,NaN,0.92,0.433964,0.024869,NaN,False,1.0,0.073007,True,84.6,2,minor_issue
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,CAND_0099996,NaN,0.32,0.336643,0.098616,NaN,False,1.0,NaN,NaN,41.4,0,trusted
99996,CAND_0099997,NaN,0.92,0.470643,0.040267,NaN,False,1.0,NaN,NaN,42.7,0,trusted
99997,CAND_0099998,0.361263,0.16,0.382071,0.254980,0.770,False,0.5,NaN,NaN,75.9,2,minor_issue
99998,CAND_0099999,0.490984,0.68,0.607321,0.097415,0.554,False,1.0,NaN,NaN,49.1,0,trusted


In [80]:
quality_features = [

    "profile_completeness",

    "views",

    "search_appearance",

    "saved",

    "connections",

    "completion_rate"
]

In [83]:
from sklearn.preprocessing import StandardScaler

X = df[quality_features]

X_scaled = StandardScaler().fit_transform(X)

In [84]:
from sklearn.decomposition import PCA

pca = PCA(n_components=1)

quality_score = pca.fit_transform(X_scaled)

df["quality_score"] = quality_score

In [85]:
df["quality_score"].describe()

count    1.000000e+05
mean     1.080025e-17
std      1.336318e+00
min     -3.120069e+00
25%     -8.546907e-01
50%     -2.394517e-01
75%      5.467699e-01
max      1.639319e+01
Name: quality_score, dtype: float64

In [86]:
master_df = df[
[
    "candidate_id",

    "quality_score",

    "behavior_score",

    "education_score",

    "skill_desc_similarity",

    "honeypot_score",

    "honey_pot_labels"
]
].copy()

In [88]:
master_df["forensic_rank_score"] = (

      0.35 * master_df["quality_score"]

    + 0.30 * master_df["behavior_score"]

    + 0.20 * master_df["skill_desc_similarity"]

    + 0.15 * master_df["education_score"]

    - 0.10 * master_df["honeypot_score"]

)

In [89]:
top100 = (
    master_df
    .sort_values(
        "forensic_rank_score",
        ascending=False
    )
    .head(100)
)

top100.head(20)

,candidate_id,quality_score,behavior_score,education_score,skill_desc_similarity,honeypot_score,honey_pot_labels,forensic_rank_score
30,CAND_0000031,7.405954,0.653590,1.0,0.212809,0,trusted,2.980723
81,CAND_0000082,3.051655,0.587741,0.5,0.094025,4,suspicious,0.938207
62,CAND_0000063,1.586213,0.598290,1.0,0.246967,0,trusted,0.934055
53,CAND_0000054,0.856366,0.470941,1.0,0.233326,0,trusted,0.637675
0,CAND_0000001,0.643806,0.489693,1.0,0.156749,1,minor_issue,0.453590
87,CAND_0000088,0.301155,0.396510,1.0,0.208540,0,trusted,0.416065
50,CAND_0000051,0.462023,0.581927,1.0,0.137119,1,minor_issue,0.413710
17,CAND_0000018,0.568241,0.250412,1.0,0.199216,1,minor_issue,0.363851
35,CAND_0000036,0.533972,0.483935,1.0,0.091147,2,minor_issue,0.300300
84,CAND_0000085,0.160012,0.388566,1.0,0.184668,1,minor_issue,0.259507


In [90]:
from sklearn.preprocessing import MinMaxScaler

cols = [

    "quality_score",
    "behavior_score",
    "education_score",
    "skill_desc_similarity"

]

scaler = MinMaxScaler()

master_df[
    [f"{c}_norm" for c in cols]
] = scaler.fit_transform(
    master_df[cols]
)

In [95]:
master_df["forensic_rank_score"] = (

      0.35 * master_df["quality_score_norm"]

    + 0.30 * master_df["behavior_score_norm"]

    + 0.20 * master_df["skill_desc_similarity_norm"]

    + 0.15 * master_df["education_score_norm"]

    - 0.25 * (
        master_df["honeypot_score"] /
        master_df["honeypot_score"].max()
      )

)

In [97]:
top100 = (
    master_df
    .sort_values(
        "forensic_rank_score",
        ascending=False
    )
    .head(100)
)

top100.head(30)

,candidate_id,quality_score,behavior_score,education_score,skill_desc_similarity,honeypot_score,honey_pot_labels,forensic_rank_score,quality_score_norm,behavior_score_norm,education_score_norm,skill_desc_similarity_norm
30,CAND_0000031,7.405954,0.653590,1.0,0.212809,0,trusted,0.678438,0.539429,0.763995,1.0,0.552195
62,CAND_0000063,1.586213,0.598290,1.0,0.246967,0,trusted,0.570055,0.241184,0.687269,1.0,0.647301
53,CAND_0000054,0.856366,0.470941,1.0,0.233326,0,trusted,0.496361,0.203781,0.510579,1.0,0.609322
55,CAND_0000056,-0.619582,0.336782,1.0,0.371983,0,trusted,0.491259,0.128143,0.324440,1.0,0.995386
50,CAND_0000051,0.462023,0.581927,1.0,0.137119,1,minor_issue,0.450660,0.183572,0.664567,1.0,0.341449
87,CAND_0000088,0.301155,0.396510,1.0,0.208540,0,trusted,0.441620,0.175328,0.407311,1.0,0.540308
67,CAND_0000068,-0.248863,0.553346,0.5,0.332590,2,minor_issue,0.428614,0.147141,0.624912,0.5,0.885702
0,CAND_0000001,0.643806,0.489693,1.0,0.156749,1,minor_issue,0.426461,0.192888,0.536597,1.0,0.396108
86,CAND_0000087,-0.742568,0.476295,1.0,0.180390,1,minor_issue,0.409182,0.121840,0.518008,1.0,0.461929
26,CAND_0000027,0.069137,0.531406,0.5,0.303020,2,minor_issue,0.408719,0.163438,0.594472,0.5,0.803371


In [101]:
top20 = (
    master_df
    .sort_values(
        "forensic_rank_score",
        ascending=False
    )
    .head(100)
)

top20["honey_pot_labels"].value_counts()

honey_pot_labels
minor_issue    46
suspicious     30
trusted        22
high_risk       2
Name: count, dtype: int64

In [102]:
def trust_multiplier(label):

    if label == "trusted":
        return 1.00

    elif label == "minor_issue":
        return 0.90

    elif label == "suspicious":
        return 0.70

    elif label == "high_risk":
        return 0.40

    return 1.00

In [103]:
top50_ids = (
    master_df
    .sort_values(
        "forensic_rank_score",
        ascending=False
    )
    .head(50)["candidate_id"]
)

In [104]:
top50_ids

30    CAND_0000031
62    CAND_0000063
53    CAND_0000054
55    CAND_0000056
50    CAND_0000051
87    CAND_0000088
67    CAND_0000068
0     CAND_0000001
86    CAND_0000087
26    CAND_0000027
84    CAND_0000085
47    CAND_0000048
57    CAND_0000058
35    CAND_0000036
17    CAND_0000018
46    CAND_0000047
31    CAND_0000032
82    CAND_0000083
36    CAND_0000037
40    CAND_0000041
81    CAND_0000082
39    CAND_0000040
93    CAND_0000094
68    CAND_0000069
83    CAND_0000084
85    CAND_0000086
45    CAND_0000046
16    CAND_0000017
69    CAND_0000070
70    CAND_0000071
27    CAND_0000028
73    CAND_0000074
72    CAND_0000073
8     CAND_0000009
74    CAND_0000075
12    CAND_0000013
91    CAND_0000092
94    CAND_0000095
1     CAND_0000002
2     CAND_0000003
3     CAND_0000004
4     CAND_0000005
5     CAND_0000006
6     CAND_0000007
7     CAND_0000008
9     CAND_0000010
10    CAND_0000011
11    CAND_0000012
13    CAND_0000014
14    CAND_0000015
Name: candidate_id, dtype: str